In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# MIMIC PRESCRIPTION PROCESSING
# Cell 1: Project setup + data inspection
# ============================================================

from pathlib import Path
import pandas as pd
import json
import os

# ------------------------------------------------------------
# 1. Locate project directory
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd()

print("Current working directory:")
print(PROJECT_DIR)

print("\nProject contents:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" -", p.name)


# ------------------------------------------------------------
# 2. Check important folders
# ------------------------------------------------------------

DATA_DIR = PROJECT_DIR / "data"

print("\nData directory:")
print(DATA_DIR)

if DATA_DIR.exists():
    print("\nFiles inside data/:")
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)
else:
    print("WARNING: data/ directory was not found.")


# ------------------------------------------------------------
# 3. Check verified dose table from Notebook 01
# ------------------------------------------------------------

DOSE_TABLE_PATH = DATA_DIR / "dose_table.json"
print("\nChecking verified dose table...")
if DOSE_TABLE_PATH.exists():

    with open(DOSE_TABLE_PATH, "r", encoding="utf-8") as f:
        dose_table = json.load(f)

    print("dose_table.json FOUND")
    print("Verified limits:", len(dose_table.get("limits", [])))
else:
    print("WARNING: dose_table.json was NOT found.")


# ------------------------------------------------------------
# 4. Search for MIMIC prescription files
# ------------------------------------------------------------

print("\nSearching for MIMIC prescription files...")

possible_files = []

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        filename = file.lower()

        if (
            "prescription" in filename
            or filename == "prescriptions.csv"
            or filename == "prescriptions.csv.gz"
            or filename == "prescriptions.csv.gz"
        ):
            possible_files.append(Path(root) / file)

if possible_files:
    print("Possible prescription files found:")

    for p in possible_files:
        print(" -", p)
else:
    print("No prescription file found automatically.")

# ------------------------------------------------------------
# 5. Check common MIMIC filenames
# ------------------------------------------------------------

common_names = [
    "PRESCRIPTIONS.csv",
    "PRESCRIPTIONS.csv.gz",
    "prescriptions.csv",
    "prescriptions.csv.gz",]

print("\nChecking common MIMIC filenames:")

for name in common_names:
    matches = list(PROJECT_DIR.rglob(name))

    if matches:
        for match in matches:
            print("FOUND:", match)

print("\n============================================================")
print("CELL 1 COMPLETE")
print("============================================================")

In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 2: Load MIMIC-III Demo tables
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. MIMIC dataset location
# ------------------------------------------------------------

DATA_DIR = Path.home() / "dataset"

print("MIMIC DATA DIRECTORY")

print(DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError( f"MIMIC dataset directory not found: {DATA_DIR}"    )

print("\nDataset directory exists ✅")

# ------------------------------------------------------------
# 2. Expected MIMIC files
# ------------------------------------------------------------

required_files = [
    "PRESCRIPTIONS.csv",
    "DIAGNOSES_ICD.csv",
    "D_ICD_DIAGNOSES.csv",
    "ADMISSIONS.csv",
    "PATIENTS.csv",
]

print("\nChecking required files:")
missing = []
for filename in required_files:
    path = DATA_DIR / filename
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"  ✅ {filename}  ({size_mb:.2f} MB)")
    else:
        print(f"  ❌ {filename}  NOT FOUND")
        missing.append(filename)

if missing:
    raise FileNotFoundError( "Missing MIMIC files: " + ", ".join(missing)    )
# ------------------------------------------------------------
# 3. Load PRESCRIPTIONS
# ------------------------------------------------------------

print("LOADING PRESCRIPTIONS")

PRESCRIPTIONS_PATH = DATA_DIR / "PRESCRIPTIONS.csv"
prescriptions = pd.read_csv( PRESCRIPTIONS_PATH, low_memory=False)

print("Loaded successfully ✅")
print("Shape:", prescriptions.shape)

print("\nColumns:")
for col in prescriptions.columns:
    print(" -", col)

# ------------------------------------------------------------
# 4. Load PATIENTS
# ------------------------------------------------------------

print("LOADING PATIENTS")

patients = pd.read_csv( DATA_DIR / "PATIENTS.csv", low_memory=False)

print("Loaded successfully ✅")
print("Shape:", patients.shape)

print("\nColumns:")
for col in patients.columns:
    print(" -", col)

# ------------------------------------------------------------
# 5. Load ADMISSIONS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("LOADING ADMISSIONS")
print("=" * 60)

admissions = pd.read_csv(DATA_DIR / "ADMISSIONS.csv", low_memory=False)

print("Loaded successfully ✅")
print("Shape:", admissions.shape)

print("\nColumns:")
for col in admissions.columns:
    print(" -", col)
# ------------------------------------------------------------
# 6. Quick prescription preview
# ------------------------------------------------------------

print("PRESCRIPTIONS PREVIEW")

display(prescriptions.head())

print("\n" + "=" * 60)
print("CELL 2 COMPLETE")

In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 3: Calculate age at admission
# Safe version for MIMIC de-identified dates
# ============================================================

import pandas as pd
import numpy as np

print("CALCULATING AGE AT ADMISSION")

# ------------------------------------------------------------
# 1. Convert dates
# ------------------------------------------------------------

patients["dob"] = pd.to_datetime(
    patients["dob"],  errors="coerce")
admissions["admittime"] = pd.to_datetime(admissions["admittime"],  errors="coerce")
print("Date conversion complete ✅")
# ------------------------------------------------------------
# 2. Create patient -> DOB lookup
# ------------------------------------------------------------
patient_dates = patients[ ["subject_id", "dob"]].drop_duplicates( subset=["subject_id"])

print("Unique patients with DOB:", len(patient_dates))
# ------------------------------------------------------------
# 3. Attach DOB to admissions
# ------------------------------------------------------------
admission_age = admissions[["subject_id", "hadm_id", "admittime"]].copy()

admission_age = admission_age.merge( patient_dates, on="subject_id", how="left")
# ------------------------------------------------------------
# 4. Calculate age WITHOUT datetime subtraction
# ------------------------------------------------------------
# This avoids pandas timedelta overflow caused by some
# de-identified MIMIC dates.
# ------------------------------------------------------------

admission_age["age_years"] = ( admission_age["admittime"].dt.year - admission_age["dob"].dt.year)

# Adjust if the birthday had not occurred yet in the
# admission year.

birthday_not_reached = (
    (admission_age["admittime"].dt.month < admission_age["dob"].dt.month)  |  ( (admission_age["admittime"].dt.month == admission_age["dob"].dt.month)
              &  (admission_age["admittime"].dt.day < admission_age["dob"].dt.day) ))

admission_age.loc[ birthday_not_reached, "age_years"] -= 1

# ------------------------------------------------------------
# 5. Basic diagnostics
# ------------------------------------------------------------

print("\nAge calculation complete ✅")

print("\nMissing DOB:")
print(admission_age["dob"].isna().sum())

print("\nMissing admission time:")
print(admission_age["admittime"].isna().sum())

print("\nMissing calculated age:")
print(admission_age["age_years"].isna().sum())

# ------------------------------------------------------------
# 6. Age statistics
# ------------------------------------------------------------

print("AGE DISTRIBUTION")

print(admission_age["age_years"].describe())

# ------------------------------------------------------------
# 7. Youngest admissions
# ------------------------------------------------------------

print("\nYoungest admissions:")

display(
    admission_age
    .sort_values("age_years")
    [["subject_id", "hadm_id", "dob", "admittime", "age_years"]]
    .head(10))

# ------------------------------------------------------------
# 8. Oldest admissions
# ------------------------------------------------------------

print("\nOldest admissions:")

display( admission_age
    .sort_values("age_years", ascending=False) [["subject_id", "hadm_id", "dob", "admittime", "age_years"]] .head(10))

# ------------------------------------------------------------
# 9. Adult / non-adult diagnostic
# ------------------------------------------------------------

adult_mask = admission_age["age_years"] >= 18

print("ADULT / NON-ADULT CHECK")

print("Adult admissions (>=18):", adult_mask.sum())
print("Non-adult admissions (<18):", (~adult_mask).sum())


print("CELL 3 COMPLETE")


In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 3.5: Inspect suspicious MIMIC ages
# ============================================================

print("SUSPICIOUS AGE CHECK")

suspicious = admission_age[
    (admission_age["age_years"] < 0)
    | (admission_age["age_years"] > 100)].copy()

print( "Admissions with age < 0 or > 100:", len(suspicious))
print("\nSuspicious admissions:")
display(
    suspicious[
        [
            "subject_id",
            "hadm_id",
            "dob",
            "admittime",
            "age_years"
        ]
    ] .sort_values("age_years"))

print("AGE FREQUENCY TABLE")

display(admission_age["age_years"].value_counts() .sort_index())

print("CELL 3.5 COMPLETE")


In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 4: Create adult admission table
# ============================================================

print("=" * 60)
print("CREATING ADULT ADMISSION TABLE")
print("=" * 60)

# ------------------------------------------------------------
# 1. Start from calculated admission_age
# ------------------------------------------------------------
adult_admissions = admission_age.copy()
# ------------------------------------------------------------
# 2. Mark de-identified ages
# ------------------------------------------------------------
adult_admissions["age_deidentified"] = (adult_admissions["age_years"] > 100)
# ------------------------------------------------------------
# 3. Adult eligibility
# ------------------------------------------------------------
#
# Normal adult:
#     age >= 18
#
# De-identified MIMIC age:
#     age > 100
#
# We retain these as adult records but flag them.
# ------------------------------------------------------------

adult_admissions["is_adult"] = ( (adult_admissions["age_years"] >= 18)  | (adult_admissions["age_deidentified"]))

# ------------------------------------------------------------
# 4. Remove the one clearly pediatric admission
# ------------------------------------------------------------

adult_admissions = adult_admissions[ adult_admissions["is_adult"]].copy()

# ------------------------------------------------------------
# 5. Diagnostic summary
# ------------------------------------------------------------

print("\nAdult admissions retained:")
print(len(adult_admissions))

print("\nDe-identified-age admissions retained:")
print(    adult_admissions["age_deidentified"].sum())

print("\nNormal adult admissions:")
print(    (  adult_admissions["age_years"].between(18, 100)).sum())

print("\nPediatric admissions excluded:")
print(    (admission_age["age_years"] < 18).sum())


# ------------------------------------------------------------
# 6. Age distribution after filtering
# ------------------------------------------------------------

print("RETAINED AGE DISTRIBUTION")

display( adult_admissions[ ["age_years", "age_deidentified"] ] .value_counts().sort_index())

# ------------------------------------------------------------
# 7. Preview
# ------------------------------------------------------------

print("\nAdult admission preview:")
display(
    adult_admissions[
        [
            "subject_id",
            "hadm_id",
            "dob",
            "admittime",
            "age_years",
            "age_deidentified",
            "is_adult"
        ] ].head(10))

print("CELL 4 COMPLETE")

In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 5: Filter prescriptions to adult admissions
# ============================================================

print("FILTERING PRESCRIPTIONS TO ADULT ADMISSIONS")

# ------------------------------------------------------------
# 1. Keep only the admission information we need
# ------------------------------------------------------------

adult_admission_info = adult_admissions[
    [
        "subject_id",
        "hadm_id",
        "age_years",
        "age_deidentified"
    ]].copy()


# ------------------------------------------------------------
# 2. Merge admission age information into prescriptions
# ------------------------------------------------------------

prescriptions_with_age = prescriptions.merge(adult_admission_info, on=["subject_id", "hadm_id"], how="inner")

# ------------------------------------------------------------
# 3. Basic counts
# ------------------------------------------------------------

print("\nOriginal prescriptions:")
print(len(prescriptions))

print("\nAdult prescriptions:")
print(len(prescriptions_with_age))

print("\nPrescriptions removed:")
print( len(prescriptions) - len(prescriptions_with_age))


# ------------------------------------------------------------
# 4. Check patients and admissions represented
# ------------------------------------------------------------

print("\nUnique patients:")
print( prescriptions_with_age["subject_id"].nunique())

print("\nUnique admissions:")
print( prescriptions_with_age["hadm_id"].nunique())


# ------------------------------------------------------------
# 5. Check de-identified age prescriptions
# ------------------------------------------------------------

print("\nPrescriptions belonging to de-identified-age admissions:")
print(prescriptions_with_age["age_deidentified"].sum())

# ------------------------------------------------------------
# 6. Check age distribution of prescriptions
# ------------------------------------------------------------

print("AGE DISTRIBUTION OF PRESCRIPTIONS")

display(
    prescriptions_with_age["age_years"]
    .value_counts()
    .sort_index())

# ------------------------------------------------------------
# 7. Preview important prescription fields
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("ADULT PRESCRIPTION PREVIEW")
print("=" * 60)

display(
    prescriptions_with_age[
        [
            "subject_id",
            "hadm_id",
            "drug",
            "drug_name_generic",
            "prod_strength",
            "dose_val_rx",
            "dose_unit_rx",
            "route",
            "age_years",
            "age_deidentified"
        ]
    ].head(15))
# ------------------------------------------------------------
# 8. Store as the working adult prescription dataframe
# ------------------------------------------------------------

adult_prescriptions = prescriptions_with_age.copy()

print("CELL 5 COMPLETE")

Cell 6: Prescription data audit

In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 6: Prescription data audit
# ============================================================

print("PRESCRIPTION DATA AUDIT")

# ------------------------------------------------------------
# 1. Missing values in important fields
# ------------------------------------------------------------

important_columns = [
    "drug",
    "drug_name_generic",
    "prod_strength",
    "dose_val_rx",
    "dose_unit_rx",
    "route",
    "startdate",
    "enddate"]

print("MISSING VALUES")
missing_summary = ( adult_prescriptions[important_columns].isna() .sum() .sort_values(ascending=False))

display(missing_summary.to_frame("missing_count"))

# ------------------------------------------------------------
# 2. Dose units
# ------------------------------------------------------------

print("DOSE UNITS")

dose_units = (
    adult_prescriptions["dose_unit_rx"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
    .value_counts())

display(dose_units.head(30).to_frame("count"))

# ------------------------------------------------------------
# 3. Routes
# ------------------------------------------------------------

print("ROUTES")

routes = (
    adult_prescriptions["route"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
    .value_counts())

display(routes.to_frame("count"))

# ------------------------------------------------------------
# 4. Drug types
# ------------------------------------------------------------

print("DRUG TYPES")

drug_types = (
    adult_prescriptions["drug_type"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
    .value_counts())

display(drug_types.to_frame("count"))
# ------------------------------------------------------------
# 5. Most common drugs
# ------------------------------------------------------------

print("TOP DRUG NAMES")

top_drugs = (
    adult_prescriptions["drug_name_generic"]
    .fillna("<MISSING>")
    .astype(str)
    .str.strip()
    .value_counts())

display(top_drugs.head(30).to_frame("count"))

# ------------------------------------------------------------
# 6. Numeric dose diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DOSE VALUE DIAGNOSTICS")
print("=" * 60)

print( "dose_val_rx dtype:",    adult_prescriptions["dose_val_rx"].dtype)
print( "Numeric dose values:", pd.to_numeric( adult_prescriptions["dose_val_rx"], errors="coerce" ).notna().sum())
print( "Non-numeric dose values:", pd.to_numeric(  adult_prescriptions["dose_val_rx"],  errors="coerce" ).isna().sum())

# ------------------------------------------------------------
# 7. Formulation examples
# ------------------------------------------------------------

print("PRODUCT STRENGTH EXAMPLES")

display(
    adult_prescriptions[
        [
            "drug_name_generic",
            "prod_strength",
            "dose_val_rx",
            "dose_unit_rx",
            "route"
        ] ] .drop_duplicates() .head(30))


# ------------------------------------------------------------
# 8. Date columns
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PRESCRIPTION DATE TYPES")
print("=" * 60)

print("startdate dtype:", adult_prescriptions["startdate"].dtype)
print("enddate dtype:", adult_prescriptions["enddate"].dtype)

print("CELL 6 COMPLETE")

In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 6B: Remaining prescription audit
# ============================================================

print("ROUTES")


display(
    adult_prescriptions["route"]
    .astype(str)
    .str.strip()
    .value_counts()
    .to_frame("count"))

print("DRUG TYPES")

display(
    adult_prescriptions["drug_type"]
    .astype(str)
    .str.strip()
    .value_counts()
    .to_frame("count"))

print("TOP DRUG NAMES")

display(
    adult_prescriptions["drug"]
    .astype(str)
    .str.strip()
    .value_counts()
    .head(40)
    .to_frame("count"))

print("DOSE VALUE DIAGNOSTICS")

numeric_dose = pd.to_numeric(adult_prescriptions["dose_val_rx"],  errors="coerce")

print("Total prescription rows:", len(adult_prescriptions))
print("Numeric dose values:", numeric_dose.notna().sum())
print("Non-numeric dose values:", numeric_dose.isna().sum())

print("\nDose value statistics:")
display(numeric_dose.describe())

print("CELL 6B COMPLETE")

In [ ]:
# ============================================================
# CELL 6C: FINAL ROUTE + DRUG TYPE AUDIT
# ============================================================

display(
    adult_prescriptions["route"]
    .astype(str)
    .str.strip()
    .value_counts()
    .to_frame("count"))


print("DRUG TYPES")

display(
    adult_prescriptions["drug_type"]
    .astype(str)
    .str.strip()
    .value_counts()
    .to_frame("count"))

print("NON-NUMERIC DOSE EXAMPLES")

numeric_dose = pd.to_numeric( adult_prescriptions["dose_val_rx"], errors="coerce")

display(
    adult_prescriptions.loc[
        numeric_dose.isna(),
        [
            "drug",
            "drug_name_generic",
            "prod_strength",
            "dose_val_rx",
            "dose_unit_rx",
            "route"
        ]
    ].head(30))

print("CELL 6C COMPLETE")

### Cell 7: Controlled normalization

In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# Cell 7: Controlled prescription normalization
# ============================================================

print("CONTROLLED PRESCRIPTION NORMALIZATION")

df = adult_prescriptions.copy()

# ------------------------------------------------------------
# 1. Drug-name fallback
# ------------------------------------------------------------
#
# Generic name is preferred.
# If unavailable, use the original drug field.
# We preserve both raw fields.
# ------------------------------------------------------------

df["drug_name_generic_raw"] = df["drug_name_generic"]

df["drug_raw"] = (    df["drug"] .astype(str) str.strip())
df["drug_name_normalized"] = (    df["drug_name_generic"]    .fillna(df["drug"])    .astype(str)    .str.strip())

# Protect against literal missing strings
df.loc[df["drug_name_normalized"].isin( ["", "nan", "<MISSING>"]    ),    "drug_name_normalized"] = df["drug_raw"]
# ------------------------------------------------------------
# 2. Dose value
# ------------------------------------------------------------

df["dose_value_raw"] = df["dose_val_rx"]
df["dose_value_numeric"] = pd.to_numeric(    df["dose_val_rx"],    errors="coerce")

# ------------------------------------------------------------
# 3. Normalize dose-unit formatting
# ------------------------------------------------------------

df["dose_unit_raw"] = (
    df["dose_unit_rx"]
    .astype(str)
    .str.strip())

dose_unit_map = {
    "ml": "mL",
    "mL": "mL",

    "gm": "g",
    "g": "g",

    "mg": "mg",
    "mcg": "mcg",
    "mEq": "mEq",
    "mmol": "mmol",

    "UNIT": "unit",
    "million units": "million_unit",

    "mcg/hr": "mcg/hr",
    "mg PE": "mg_PE"
}

df["dose_unit_normalized"] = (
    df["dose_unit_raw"]
    .map(dose_unit_map)
    .fillna(df["dose_unit_raw"]))

# ------------------------------------------------------------
# 4. Preserve formulation/administration units
# ------------------------------------------------------------

formulation_units = { "TAB", "NEB", "VIAL", "Appl", "PUFF", "PKT", "DROP", "BAG","SYR", "CAP", "PTCH", "INH", "AMP", "LOZ","SPRY","TROC","dose"}

df["is_formulation_unit"] = ( df["dose_unit_raw"].isin(formulation_units))
# ------------------------------------------------------------
# 5. Route normalization
# ------------------------------------------------------------

df["route_raw"] = (
    df["route"]
    .astype(str)
    .str.strip())

route_map = {

    # Oral
    "PO": "ORAL",
    "ORAL": "ORAL",

    # IV family
    "IV": "IV",
    "IV DRIP": "IV_DRIP",
    "IV BOLUS": "IV_BOLUS",
    "IVPCA": "IV_PCA",

    # Subcutaneous
    "SC": "SUBCUTANEOUS",
    "SUBCUT": "SUBCUTANEOUS",

    # Enteral / gastric
    "PO/NG": "ORAL_OR_NG",
    "NG": "NASOGASTRIC",
    "OG": "OROGASTRIC",
    "J TUBE": "JEJUNAL_TUBE",

    # Intramuscular
    "IM": "INTRAMUSCULAR",

    # Rectal
    "PR": "RECTAL",

    # Intrathecal
    "IT": "INTRATHECAL",

    # Topical
    "TP": "TOPICAL",
    "TD": "TRANSDERMAL",

    # Sublingual
    "SL": "SUBLINGUAL",

    # Inhalation
    "IH": "INHALATION",
    "NEB": "NEBULIZED",

    # Dialysis
    "DIALYS": "DIALYSIS",

    # Ophthalmic
    "OD": "RIGHT_EYE",
    "OS": "LEFT_EYE",
    "OU": "BOTH_EYES",
    "RIGHT EYE": "RIGHT_EYE",
    "LEFT EYE": "LEFT_EYE",
    "BOTH EYES": "BOTH_EYES"
}

df["route_normalized"] = (
    df["route_raw"]
    .map(route_map)
    .fillna(df["route_raw"]))

# ------------------------------------------------------------
# 6. Flag routes requiring special handling
# ------------------------------------------------------------

standard_routes = set(route_map.values())

df["route_special"] = ~df["route_normalized"].isin(
    standard_routes)

# ------------------------------------------------------------
# 7. Dose parse status
# ------------------------------------------------------------

df["dose_parse_status"] = np.where(
    df["dose_value_numeric"].notna(),
    "numeric",
    "non_numeric")

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\nDrug-name fallback usage:")

generic_available = df["drug_name_generic"].notna().sum()

print( "Generic name available:", generic_available)
print("Generic name missing:", df["drug_name_generic"].isna().sum())
print( "Fallback to drug field:",  df["drug_name_generic"].isna().sum())

print("\nDose parsing:")
print( df["dose_parse_status"].value_counts())


print("\nNormalized dose units:")
display(
    df["dose_unit_normalized"]
    .value_counts()
    .head(30)
    .to_frame("count"))

print("\nNormalized routes:")
display(
    df["route_normalized"]
    .value_counts()
    .to_frame("count"))

print("\nSpecial/unmapped routes:")
display(
    df.loc[
        df["route_special"],
        "route_raw"
    ]
    .value_counts()
    .to_frame("count"))

# ------------------------------------------------------------
# 9. Preview normalized records
# ------------------------------------------------------------

print("NORMALIZED PRESCRIPTION PREVIEW")

display(
    df[
        [
            "drug_raw",
            "drug_name_generic_raw",
            "drug_name_normalized",
            "dose_value_raw",
            "dose_value_numeric",
            "dose_unit_raw",
            "dose_unit_normalized",
            "route_raw",
            "route_normalized",
            "age_years",
            "age_deidentified"
        ]
    ].head(20))

# ------------------------------------------------------------
# 10. Store normalized dataframe
# ------------------------------------------------------------

adult_prescriptions_normalized = df.copy()

print("CELL 7 COMPLETE")

In [ ]:
# ============================================================
# CELL 7C: Final normalization verification
# ============================================================

print("NORMALIZED ROUTES")

display(
    adult_prescriptions_normalized[
        "route_normalized" ]
    .value_counts()
    .to_frame("count"))

print("SPECIAL / UNMAPPED ROUTES")

display(
    adult_prescriptions_normalized.loc[
        adult_prescriptions_normalized["route_special"],
        "route_raw"  ] .value_counts() .to_frame("count"))

print("NON-NUMERIC DOSE EXAMPLES")

non_numeric = adult_prescriptions_normalized[ adult_prescriptions_normalized["dose_parse_status"] == "non_numeric"]
print("Non-numeric rows:", len(non_numeric))

display(
    non_numeric[
        [
            "drug_raw",
            "drug_name_normalized",
            "prod_strength",
            "dose_value_raw",
            "dose_unit_raw",
            "route_raw"
        ]
    ].head(50))

print("CELL 7C COMPLETE")

### Cell 8 : Dose Parser


In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# CELL 8: RANGE-AWARE DOSE PARSER
# ============================================================

import re
import numpy as np
import pandas as pd

print("RANGE-AWARE DOSE PARSING")

# ------------------------------------------------------------
# 1. Parser function
# ------------------------------------------------------------

def parse_dose_value(value):
    """
    Parse dose values into:
        dose_min
        dose_max
        dose_type

    Examples:
        325       -> 325, 325, single
        "325-650" -> 325, 650, range
        "1-2"     -> 1, 2, range
        "0.5"     -> 0.5, 0.5, single

    Anything ambiguous remains unparsed.
    """

    if pd.isna(value):
        return np.nan, np.nan, "missing"
    text = str(value).strip()

    if text == "":
        return np.nan, np.nan, "missing"

    # Remove commas from values such as 25,000
    text = text.replace(",", "")

    # --------------------------------------------------------
    # Single numeric value
    # --------------------------------------------------------

    try:
        number = float(text)
        return number, number, "single"
    except ValueError:
        pass
    # --------------------------------------------------------
    # Numeric range
    #
    # Accept:
    # 1-2
    # 0.5-2
    # 325-650
    # --------------------------------------------------------

    match = re.fullmatch( r"\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*", text )

    if match:

        minimum = float(match.group(1))
        maximum = float(match.group(2))

        # Sanity check
        if minimum <= maximum:
            return minimum, maximum, "range"

    # --------------------------------------------------------
    # Anything else remains unparsed
    # --------------------------------------------------------
    return np.nan, np.nan, "unparsed"

# ------------------------------------------------------------
# 2. Apply parser
# ------------------------------------------------------------

parsed = (
    adult_prescriptions_normalized["dose_value_raw"]
    .apply(parse_dose_value))

adult_prescriptions_normalized[ "dose_min"] = parsed.apply(lambda x: x[0])
adult_prescriptions_normalized[ "dose_max"] = parsed.apply(lambda x: x[1])
adult_prescriptions_normalized[ "dose_type"] = parsed.apply(lambda x: x[2])

# ------------------------------------------------------------
# 3. Verification statistics
# ------------------------------------------------------------

print("\nDose parsing results:")
display( adult_prescriptions_normalized[ "dose_type"  ] .value_counts().to_frame("count"))

# ------------------------------------------------------------
# 4. Check ranges
# ------------------------------------------------------------

ranges = adult_prescriptions_normalized[ adult_prescriptions_normalized["dose_type"] == "range"].copy()
print("\nTotal dose ranges:", len(ranges))
display(ranges[
        [
            "drug_name_normalized",
            "dose_value_raw",
            "dose_min",
            "dose_max",
            "dose_unit_normalized",
            "route_normalized"
        ] ].head(50))


# ------------------------------------------------------------
# 5. Check unparsed values
# ------------------------------------------------------------

unparsed = adult_prescriptions_normalized[ adult_prescriptions_normalized["dose_type"] == "unparsed"].copy()
print("\nTotal unparsed values:", len(unparsed))

if len(unparsed) > 0:

    display( unparsed[
            [
                "drug_name_normalized",
                "dose_value_raw",
                "dose_unit_raw",
                "prod_strength",
                "route_raw"
            ]  ].head(50)  )

# ------------------------------------------------------------
# 6. Final preview
# ------------------------------------------------------------

print("FINAL DOSE REPRESENTATION")

display(
    adult_prescriptions_normalized[
        [
            "drug_name_normalized",
            "dose_value_raw",
            "dose_min",
            "dose_max",
            "dose_unit_normalized",
            "route_normalized",
            "dose_type"
        ]
    ].head(30))

print("CELL 8 COMPLETE")

### Cell 9, create the canonical medication table
10,256 medication records are in the canonical dataset.

10,255 (99.99%) are ready for automated verification.

Only 1 record is missing at least one required field.

Your canonical representation is therefore in very good shape for the next stage.


In [ ]:
# ============================================================
# DOSE WHISPERER - NOTEBOOK 02
# CELL 9: CANONICAL MEDICATION RECORDS
# ============================================================

print("BUILDING CANONICAL MEDICATION RECORDS")

# ------------------------------------------------------------
# 0. Select canonical columns
# ------------------------------------------------------------

canonical_cols = [
    "subject_id",
    "hadm_id",
    "age_years",
    "age_deidentified",

    "drug_name_normalized",
    "drug_name_generic_raw",
    "drug_raw",
    "prod_strength",

    "dose_value_raw",
    "dose_value_numeric",
    "dose_unit_raw",
    "dose_unit_normalized",
    "is_formulation_unit",

    "route_raw","route_normalized","route_special",

    "dose_parse_status", "dose_min", "dose_max","dose_type",

    "startdate","enddate"
]

canonical_df = adult_prescriptions_normalized[canonical_cols].copy()

print("Canonical dataframe created ✅")
print("Shape:", canonical_df.shape)


# ------------------------------------------------------------
# 1. Create stable medication identity
# ------------------------------------------------------------

canonical_df["medication_key"] = (
    canonical_df["drug_name_normalized"]
    .astype(str)    .str.strip()    .str.lower())

# ------------------------------------------------------------
# 2. Create dose display string
# ------------------------------------------------------------

def make_dose_display(row):

    unit = str(row["dose_unit_normalized"])
    if row["dose_type"] == "single":
        value = row["dose_min"]
        if pd.notna(value):
            if float(value).is_integer():
                value = int(value)
            return f"{value} {unit}"
    elif row["dose_type"] == "range":
        minimum = row["dose_min"]
        maximum = row["dose_max"]
        if pd.notna(minimum) and pd.notna(maximum):
            if float(minimum).is_integer():
                minimum = int(minimum)
            if float(maximum).is_integer():
                maximum = int(maximum)
            return f"{minimum}-{maximum} {unit}"
    return "UNVERIFIED"

canonical_df["dose_display"] = canonical_df.apply(make_dose_display,  axis=1)

# ------------------------------------------------------------
# 3. Identify records suitable for automated checking
# ------------------------------------------------------------

canonical_df["verification_ready"] = (
    canonical_df["drug_name_normalized"].notna()
    & canonical_df["dose_min"].notna()
    & canonical_df["dose_max"].notna()
    & canonical_df["dose_unit_normalized"].notna()
    & canonical_df["route_normalized"].notna())

# ------------------------------------------------------------
# 4. Basic quality statistics
# ------------------------------------------------------------

print("\nCanonical records:", len(canonical_df))
print("\nVerification-ready records:")
print(    canonical_df["verification_ready"].value_counts())

print("\nDose types:")
display( canonical_df["dose_type"] .value_counts()  .to_frame("count"))

# ------------------------------------------------------------
# 5. Medication frequency
# ------------------------------------------------------------

print("\nTop medications:")
display( canonical_df["drug_name_normalized"]    .value_counts()    .head(30)    .to_frame("count"))

# ------------------------------------------------------------
# 6. Preview canonical representation
# ------------------------------------------------------------

print("\nCANONICAL MEDICATION PREVIEW")
display(    canonical_df[
        [
            "subject_id",
            "hadm_id",
            "drug_name_normalized",
            "prod_strength",
            "dose_display",
            "dose_type",
            "route_normalized",
            "age_years",
            "verification_ready"
        ]
    ].head(30))

# ------------------------------------------------------------
# 7. Save in memory for next notebook cells
# ------------------------------------------------------------

dose_whisperer_records = canonical_df.copy()

print("CELL 9 COMPLETE")
